# G1 Academy Bonus - Task 7: SLAM operation and map visualization (using the wrapper)

## Introduction
The full SLAM lifecycle through `sdk_wrapper.G1` -- no native `SlamRpc`, no DDS subscribers to wire up. You build a map by driving the robot with the **remote controller**, save it, relocalize against it, save named points, and navigate back to them. The finished reference for all of this is `academy/visualizations/slam_web_app.py`; this notebook does the same calls step by step.

**Using Codex/AI for this task:** `start_mapping`/`stop_mapping`/`relocate`/`get_slam_pose`/`navigate_to_point`/`pause_nav`/`resume_nav` are finished, documented methods on `sdk_wrapper.G1`. Paste a signature into Codex if a cell is not obvious, then read what it produced before running it against the robot.

In [ ]:
import sys
sys.path.append("..")
from sdk_wrapper import G1

g1 = G1(iface="eth0", domain_id=0)

## Task 1 - Build and save a map
`g1.start_mapping(slam_type="indoor")` starts building a map from the LiDAR. Run the start cell, then **drive the robot around the whole space with the hand-held remote controller** (not `loco_move`) so the LiDAR sees every wall; return near where you started. When the map looks complete, run the stop/save cell.

`g1.stop_mapping(save_path=MAP_PATH)` is what actually **saves** the map (the `end_mapping` RPC) so you can relocalize and navigate on it afterward. Use a full path the mainboard can write, e.g. `/home/unitree/test.pcd` -- a bare name like `"my_map"` fails with `errorCode 12, "The directory is illegal."`. Calling `stop_mapping()` with *no* path instead runs `close_slam`, which stops SLAM **without saving** -- an unsaved map is why relocation later fails with `errorCode 509`.

In [ ]:
MAP_PATH = "/home/unitree/test.pcd"  # a full path the mainboard can write; "my_map" -> errorCode 12
g1.start_mapping(slam_type="indoor")

### 🎮 Now drive the robot around with the remote controller
Keep `start_mapping()` running and walk the robot around the entire space with the hand-held remote, covering all the walls and returning near your starting point. Watch the map fill in (Task 4 below, or `academy/visualizations/slam_web_app.py`). When the map is complete, run the next cell to save it.

In [ ]:
g1.stop_mapping(save_path=MAP_PATH)  # end_mapping: saves the map to MAP_PATH so relocate() can load it

## Task 2 - Relocalize and save named points
Task 1 **saved** the map to `MAP_PATH` (that is what `stop_mapping(save_path=...)` does -- the no-arg `stop_mapping()` only *closes* SLAM without saving). Now relocalize against it before navigating.

- `g1.relocate(map_path=MAP_PATH, pose=(x, y, yaw))` loads the saved map and tells SLAM where the robot currently is. Pass an explicit pose from `get_slam_pose()` -- the reference `slam_web_app.py` does the same, so it never silently relocates to `(0, 0, 0)`.
- **`relocate()` failure codes:** `3104` is a transient RPC timeout (retry). `509 "The current location matching degree is low."` means the live scan doesn't match the saved map -- stand the robot in a well-mapped area, ideally near where you started mapping, and retry. Both are handled by `relocate_with_retry` below.
- `get_slam_pose()` is `None` until a valid pose arrives; `wait_for_slam_pose` polls for one, and `add_point` fails with a clear message if none comes.

Save named points as a JSON dict `{name: [x, y, yaw]}` keyed off `g1.get_slam_pose()` -- exactly the file `g1.navigate_to_point(name, points_path="slam_points.json")` reads.

In [ ]:
import json
import time
from pathlib import Path

POINTS_PATH = "slam_points.json"

def wait_for_slam_pose(g1, timeout_s=15, poll_s=0.5):
    """Poll until SLAM reports a valid (x, y, yaw), or return None after timeout_s."""
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        pose = g1.get_slam_pose()
        if pose is not None:
            return pose
        time.sleep(poll_s)
    return None

def relocate_with_retry(g1, map_path=MAP_PATH, pose=None, retries=8, delay=2):
    """Relocalize against the saved map, passing an explicit pose (like slam_web_app.py)
    so it never silently relocates to (0, 0, 0). Retries transient failures and surfaces
    the two you will actually see: 3104 (RPC timeout) and 509 (low map match)."""
    for attempt in range(1, retries + 1):
        current = pose or wait_for_slam_pose(g1)
        if current is None:
            print(f"relocate attempt {attempt}/{retries}: no SLAM pose yet")
            time.sleep(delay)
            continue
        result = g1.relocate(map_path=map_path, pose=tuple(current))
        if result.get("code") == 0:
            return result
        try:
            info = json.loads(result.get("raw") or "{}")
        except Exception:
            info = {}
        code = info.get("errorCode")
        hint = {509: " -- stand the robot in a mapped area, near where you started mapping",
                3104: " -- RPC timeout"}.get(code, "")
        print(f"relocate attempt {attempt}/{retries}: errorCode={code} {info.get('info', '')}{hint}")
        if attempt == retries:
            return result
        time.sleep(delay)

def add_point(g1, name, points_path=POINTS_PATH):
    pose = wait_for_slam_pose(g1)
    if pose is None:
        raise RuntimeError("No SLAM pose yet -- relocate() must reach code 0 and localize first.")
    path = Path(points_path)
    points = json.loads(path.read_text()) if path.exists() else {}
    points[name] = list(pose)
    path.write_text(json.dumps(points))
    return points[name]

def remove_point(name, points_path=POINTS_PATH):
    path = Path(points_path)
    points = json.loads(path.read_text()) if path.exists() else {}
    points.pop(name, None)
    path.write_text(json.dumps(points))

In [ ]:
# TODO: relocate against the saved map, then save the named points you drove to.
#   1. relocate_with_retry(g1) and confirm it returns code 0
#      (relocate() can return 3104 = RPC timeout, or 509 = low map match -> reposition & retry).
#   2. add_point(g1, "pickup") for each spot you walked to with the remote.
# get_slam_pose() is None until localization converges, so add_point waits for a valid pose first.
raise NotImplementedError("Complete this task section")

## Task 3 - Navigate to a saved point
`g1.navigate_to_point(name, points_path=..., timeout_s=120.0)` reads the named pose, sends it, and blocks until arrival or timeout, returning a dict with `code`/`arrived`/`pose`/`notice`. `g1.pause_nav()` / `g1.resume_nav()` interrupt and continue an in-progress navigation; `g1.loco_stop()` is the immediate full stop, including mid-navigation.

In [ ]:
# TODO: navigate_to_point() to one of the points you saved, printing the result dict
# (code / arrived / pose / notice).
raise NotImplementedError("Complete this task section")

## Task 4 - See the map
`g1.get_point_cloud()` returns the latest SLAM point cloud as a list of `(x, y, z)` points -- capture and plot it *while mapping/relocation is running*. A 2-D scatter of `x`/`y` is enough to recognize the room outline.

Two finished references you do not need to rebuild: `academy/visualizations/slam_web_app.py` (live map + robot pose in the browser, on port 8060) and `academy/visualizations/zmp_viz.py` (a live balance/ZMP plot). Run either directly, or paste it into Codex and ask for a trimmed-down version.

### Safety
Run no command cell until the subscriber state is fresh, controller ownership is known, the space is clear, and a damp path is available. Code is not invoked automatically.